In [ ]:
# premierment upload dataset:
from google.colab import files
uploaded = files.upload()

In [ ]:
# deuxiemment load dataset:
import pandas as pd
# le nom de votre fichier:
filename = list(uploaded.keys())[0]
# pour lire:
df = pd.read_csv(filename)
# pour voir les premier 5 ligne:
df.head()
# pour voir le type de cgaque column:
df.info()

In [ ]:
# cleaning de dataset:
# 1️ Supprimer les lignes sans CustomerID
df = df.dropna(subset=['CustomerID'])

# 2️ Supprimer les lignes sans Description (optionnel)
df = df.dropna(subset=['Description'])

# 3️ Vérifier et convertir les types des colonnes numériques
df['Quantity'] = df['Quantity'].astype(int)
df['UnitPrice'] = df['UnitPrice'].astype(float)
df['CustomerID'] = df['CustomerID'].astype(int)

# 4️ Supprimer les lignes avec Quantity ≤ 0 ou UnitPrice ≤ 0
df = df[(df['Quantity'] > 0) & (df['UnitPrice'] > 0)]

# 5️ Convertir InvoiceDate au format datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

# 6️ Afficher les informations après nettoyage
# df.head()      # Affiche les 5 premières lignes
df.info()        # Résumé des colonnes et types de données
df.shape         # Dimensions du DataFrame (lignes, colonnes)
# df.describe()  # Statistiques descriptives (moyenne, max, min, etc.)


In [ ]:
# exploratory data analysis (EDA) + visualisation:
#il ne donne une idee 3la tawzi3 dyal mabi3at:(min, max, mean, std) pourquoi => (c'est important avant le faire la segmentation):
df.describe()
# histograme dyal quantity et unitprice:

import matplotlib.pyplot as plt

# Histogram  pour Quantity
plt.figure(figsize=(10,4))
plt.hist(df['Quantity'], bins=50, color='skyblue')
plt.title('Distribution of Quantity')
plt.xlabel('Quantity')
plt.ylabel('Frequency')
plt.show()

# Histogram pour UnitPrice
plt.figure(figsize=(10,4))
plt.hist(df['UnitPrice'], bins=50, color='salmon')
plt.title('Distribution of UnitPrice')
plt.xlabel('UnitPrice')
plt.ylabel('Frequency')
plt.show()



In [ ]:
# le nombre de client en fonction de votre country (3adad dyal les client 3la hasab lblassa):

df['Country'].value_counts().head(10).plot(kind='bar', figsize=(10,4), color='green')
plt.title('Top 10 Countries by Number of Customers')
plt.xlabel('Country')
plt.ylabel('Number of Transactions')
plt.show()

In [ ]:
# segmentation avec K-Means clustering(ML):==> (on va faire le k-means pour diviser les client par des segments selon a votre behavior)

# choisir des colomn pour la segmentation ( quantity, unitprice):

# nous ona besoin de faire la segmentation seon le client donc il faut voir le totalamount de chaque client:
#  calculer TotalAmount
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

# Aggregate per CustomerID
customer_df = df.groupby('CustomerID').agg({
    'Quantity':'sum',
    'TotalAmount':'sum'
}).reset_index()

customer_df.head()





In [ ]:
# standarisation (avant le k-Means)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_df[['Quantity','TotalAmount']])

# en aplique mainteneat le k-means:

from sklearn.cluster import KMeans

# en choisir 3 clusters par exemple
kmeans = KMeans(n_clusters=3, random_state=42)
customer_df['Segment'] = kmeans.fit_predict(X_scaled)

customer_df.head()

In [ ]:
# visaualisation de segment:
plt.figure(figsize=(10,6))
plt.scatter(customer_df['Quantity'], customer_df['TotalAmount'], c=customer_df['Segment'], cmap='viridis')
plt.xlabel('Quantity')
plt.ylabel('TotalAmount')
plt.title('Customer Segmentation')
plt.show()

In [ ]:
# pour voir chaque segment achno fih :
# statistique de chaque Segment
customer_df.groupby('Segment').agg({
    'Quantity':'mean',
    'TotalAmount':'mean',
    'CustomerID':'count'
}).rename(columns={'CustomerID':'NumCustomers'})

In [ ]:
# Creer le Dashboard:

customer_df.to_csv("segment.csv",index=False)



In [ ]:
from google.colab import files
files.download('segment.csv')
